In [17]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [18]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [19]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [20]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [21]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [22]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [23]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [24]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [25]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [26]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-02-17 20:03:...|        Cliegg Lars| 62|https://www.swapi...|{"height": "183",...|
|2025-02-17 20:03:...|  Poggle the Lesser| 63|https://www.swapi...|{"height": "183",...|
|2025-02-17 20:03:...|    Luminara Unduli| 64|https://www.swapi...|{"height": "170",...|
|2025-02-17 20:03:...|      Barriss Offee| 65|https://www.swapi...|{"height": "166",...|
|2025-02-17 20:03:...|              Dormé| 66|https://www.swapi...|{"height": "165",...|
|2025-02-17 20:03:...|              Dooku| 67|https://www.swapi...|{"height": "193",...|
|2025-02-17 20:03:...|Bail Prestor Organa| 68|https://www.swapi...|{"height": "191",...|
|2025-02-17 20:03:...|         Jango Fett| 69|https://www.swapi...|{"height": "183",...|
|2025-02

In [27]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(truncate=False)

No. Rows: 60
+--------------------------+-----------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name       |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+-----------+---+-----------

In [28]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [29]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [30]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("uid <= '25'")

    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf


silver_instance = StarWarsSilver(
    spark, catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

In [31]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

In [32]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 17
+--------------------------+--------------------------+---------------------+---+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |properties                                                                                                                                                                                                                |
+--------------------------+--------------------------+---------------------+---+------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [33]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 18
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                       |
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2025-02-17 20:04:

# 6 Clean Up

In [34]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]